In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the `citibike` package importable when this runs from the project root.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from citibike import dataset  # noqa: E402

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [ ]:
START = "2026-05-01"
END = "2026-05-08"   # one week to start; widen to a full month when ready
SOURCE_TABLE = "station_status"   # use station_status_pre2021 for historical months

sql = f"""
    SELECT fetched_at, station_id,
           num_bikes_available, num_ebikes_available, num_docks_available,
           num_bikes_disabled, num_docks_disabled,
           is_installed, is_renting, is_returning
    FROM {SOURCE_TABLE}
    WHERE fetched_at >= %(start)s AND fetched_at < %(end)s;
"""
df = dataset.query(sql, params={"start": START, "end": END})
print(f"Loaded {len(df):,} rows from {SOURCE_TABLE}  [{START} -> {END})")
df

c:\Users\clark\Desktop\citibike\citibike\dataset.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


Loaded 6,478,048 rows from station_status  [2026-05-01 -> 2026-05-08)


,fetched_at,station_id,num_bikes_available,num_ebikes_available,num_docks_available,num_bikes_disabled,num_docks_disabled,is_installed,is_renting,is_returning
0,2026-05-05 21:00:09+00:00,af72ab76-c6cb-4994-af96-51fa8846ecdc,0,0,0,0,0,0,0,0
1,2026-05-05 21:00:09+00:00,807e9f5a-49b6-4b7e-847b-4a60cff330ed,0,0,0,0,0,0,0,0
2,2026-05-05 21:00:09+00:00,2206778196198800034,0,0,0,0,0,0,0,0
3,2026-05-05 21:00:09+00:00,41495491-5d89-4e14-aab9-c3db04aad399,0,0,0,0,0,0,0,0
4,2026-05-05 21:00:09+00:00,1905837242740508940,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
6478043,2026-05-07 00:00:32+00:00,bd6f422b-d7ae-4d7e-9261-653fdd8e6888,25,17,1,2,0,1,1,1
6478044,2026-05-07 00:00:32+00:00,2108372903577159978,3,1,12,0,0,1,1,1
6478045,2026-05-07 00:00:32+00:00,581211b2-4e42-48f2-8a8f-5f968cb1c5df,0,0,29,3,0,1,1,1
6478046,2026-05-07 00:00:32+00:00,9d344652-976b-4c2d-bede-2ef19b0fbf13,3,1,15,0,0,1,1,1


In [ ]:
# Merge in capacity (lives in station_information, not station_status) so we can
# range-check bikes vs capacity. station_information is small (~4k rows).
info = dataset.load_station_information()[["station_id", "capacity"]]
df = df.merge(info, on="station_id", how="left")
print(f"Rows missing a capacity match (orphan station_ids): "
      f"{df['capacity'].isna().sum():,}")

c:\Users\clark\Desktop\citibike\citibike\dataset.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


Rows missing a capacity match (orphan station_ids): 0


 ## 1. Shape, dtypes, memory

In [ ]:
print(df.dtypes)
print(f"\nMemory: {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB")
print(f"Stations: {df['station_id'].nunique():,}")
print(f"Time span: {df['fetched_at'].min()} -> {df['fetched_at'].max()}")

fetched_at              datetime64[ns, UTC]
station_id                           object
num_bikes_available                   int64
num_ebikes_available                  int64
num_docks_available                   int64
num_bikes_disabled                    int64
num_docks_disabled                    int64
is_installed                          int64
is_renting                            int64
is_returning                          int64
capacity                              int64
dtype: object

Memory: 1,093.7 MB
Stations: 2,407
Time span: 2026-05-05 21:00:09+00:00 -> 2026-05-07 23:58:48+00:00


In [ ]:
dupe_mask = df.duplicated(subset=["station_id", "fetched_at"], keep=False)
print(f"Duplicate (station_id, fetched_at) rows: {dupe_mask.sum():,}")
if dupe_mask.any():
    display(df[dupe_mask].sort_values(["station_id", "fetched_at"]).head(10))

Duplicate (station_id, fetched_at) rows: 0


 ## 3. Null profile — distinguish *legit* nulls from *defects*

 `num_ebikes_available` is legitimately NULL pre-2018 (no e-bikes existed);
 nulls in `num_bikes_available` or `capacity` are defects that break the build.

In [ ]:
null_counts = df.isna().sum()
null_pct = (null_counts / len(df) * 100).round(2)
nulls = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
nulls[nulls["null_count"] > 0].sort_values("null_count", ascending=False)

,null_count,null_pct


 ## 4. Impossible / out-of-range values (silent model-corruptors)

In [ ]:
checks = {
    "negative bikes": (df["num_bikes_available"] < 0),
    "negative docks": (df["num_docks_available"] < 0),
    "bikes > capacity": (df["num_bikes_available"] > df["capacity"]),
    "capacity <= 0": (df["capacity"] <= 0),
    # rough consistency: bikes + docks shouldn't wildly exceed capacity
    "bikes+docks >> capacity": (
        (df["num_bikes_available"] + df["num_docks_available"]) > (df["capacity"] * 1.5)
    ),
}
range_report = pd.Series({name: int(mask.sum()) for name, mask in checks.items()})
print("Rows failing each range check:")
print(range_report)

Rows failing each range check:
negative bikes                 0
negative docks                 0
bikes > capacity            8650
capacity <= 0              69992
bikes+docks >> capacity     5371
dtype: int64


 ## 5. Mixed dtypes / object columns that should be numeric or boolean

 These crash numeric models. (The Kaggle loader specifically had booleans
 stored as `1.0/0.0/NaN` floats.)

In [ ]:
object_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Object/string columns: {object_cols}")
for c in ["is_installed", "is_renting", "is_returning"]:
    print(f"  {c}: dtype={df[c].dtype}, unique={df[c].dropna().unique()[:5]}")

Object/string columns: ['station_id']
  is_installed: dtype=int64, unique=[0 1]
  is_renting: dtype=int64, unique=[0 1]
  is_returning: dtype=int64, unique=[0 1]


 ## 6. Time gaps — stretches where ingestion was down

 Expected cadence ~2.5 min. Flag per-station gaps far larger than that.

In [ ]:
df_sorted = df.sort_values(["station_id", "fetched_at"])
df_sorted["gap_min"] = (
    df_sorted.groupby("station_id")["fetched_at"].diff().dt.total_seconds() / 60
)
GAP_THRESHOLD_MIN = 15  # >6x the expected 2.5-min cadence
gaps = df_sorted[df_sorted["gap_min"] > GAP_THRESHOLD_MIN]
print(f"Gaps > {GAP_THRESHOLD_MIN} min: {len(gaps):,}")
print(f"Largest gap: {df_sorted['gap_min'].max():,.0f} min")
gaps[["station_id", "fetched_at", "gap_min"]].sort_values("gap_min", ascending=False).head(10)

Gaps > 15 min: 4,812
Largest gap: 149 min


,station_id,fetched_at,gap_min
4465,00284700-9d22-42ce-8485-113fed9879c1,2026-05-05 23:29:09+00:00,149.0
2453,66dc80b4-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
3796,66dc7a7d-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
4064,66dc7b10-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
2846,66dc7ba2-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
4539,66dc7c31-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
3431,66dc7cc3-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
4538,66dc7d58-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
3102,66dc7de9-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0
4577,66dc7e75-0aca-11e7-82f6-3863bb44ef7c,2026-05-05 23:29:09+00:00,149.0


 ## 7. Stuck sensors — a value frozen for a long time looks like fake availability

In [ ]:
# Stations whose bike count never changes across the whole window = suspicious.
variance_by_station = df.groupby("station_id")["num_bikes_available"].nunique()
frozen = variance_by_station[variance_by_station == 1]
print(f"Stations with a CONSTANT bike count over the window: {len(frozen):,}")
print("(some are genuinely dead/decommissioned docks; inspect before deciding)")
frozen.head(10)

Stations with a CONSTANT bike count over the window: 102
(some are genuinely dead/decommissioned docks; inspect before deciding)


station_id
034f7f3d-e632-475a-aa01-894480d153df    1
06439006-11b6-44f0-8545-c9d39035f32a    1
067d5007-bda5-4697-b0a2-ccc2837f295c    1
0f45bcf6-7028-4584-a51e-4129847dbebc    1
1846085626867634660                     1
1856767203049599002                     1
1905394575114219798                     1
1905837242740508940                     1
1906248765431831024                     1
1929513834381415014                     1
Name: num_bikes_available, dtype: int64

 ## 8. Timezone consistency — wrong tz silently corrupts lag joins

In [ ]:
print(f"fetched_at dtype: {df['fetched_at'].dtype}")
print(f"timezone-aware: {df['fetched_at'].dt.tz is not None}")

fetched_at dtype: datetime64[ns, UTC]
timezone-aware: True


 ## Summary

 Roll the checks into one verdict so this is glanceable when looping months.

In [ ]:
summary = {
    "rows": len(df),
    "stations": df["station_id"].nunique(),
    "duplicate_rows": int(dupe_mask.sum()),
    "orphan_station_ids (no capacity)": int(df["capacity"].isna().sum()),
    "negative_values": int(checks["negative bikes"].sum() + checks["negative docks"].sum()),
    "bikes_gt_capacity": int(checks["bikes > capacity"].sum()),
    "time_gaps_gt_15min": len(gaps),
    "frozen_stations": len(frozen),
}
pd.Series(summary, name="data_quality_summary").to_frame()

,data_quality_summary
rows,6478048
stations,2407
duplicate_rows,0
orphan_station_ids (no capacity),0
negative_values,0
bikes_gt_capacity,8650
time_gaps_gt_15min,4812
frozen_stations,102
